In [ ]:
import pandas as pd
import numpy as np
import os
import uuid

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
OUTPUT_DIR = "nac_relational_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

POPULATION_TARGETS = {
    2025: 350000, 2030: 1050000, 2035: 1950000, 
    2040: 3100000, 2045: 4600000, 2050: 6500000
}

# Unique ID tracking to prevent collisions
USED_NATIONAL_IDS = set()
USED_REAL_ESTATE_IDS = set()

CAREERS = {
    'Civil Engineering': (120000, 700000), 'Accounting': (80000, 400000),
    'Medicine': (150000, 1200000), 'Software Engineering': (180000, 900000),
    'Architecture': (120000, 800000), 'Pharmacy': (90000, 450000),
    'Digital Marketing': (100000, 600000), 'Nursing': (80000, 300000),
    'Data Science': (200000, 1000000), 'Cybersecurity': (250000, 1100000),
    'Unemployed/Student': (0, 0), 'Retired': (40000, 240000)
}
MAJORS = [k for k in CAREERS.keys() if k not in ['Unemployed/Student', 'Retired']]
MAJOR_PROBS = [0.25, 0.20, 0.12, 0.10, 0.08, 0.08, 0.07, 0.05, 0.03, 0.02]

# ... [Keep previous ID Generation and Environmental Helpers here] ...

def assign_education_and_major(age):
    """Refined education logic to ensure Major is locked in at age 19."""
    if age < 4: return "None", "None", "None", "None", "None"
    elif 4 <= age <= 18:
        grade = "Kindergarten" if age <= 6 else ("Primary" if age <= 12 else ("Preparatory" if age <= 15 else "Secondary"))
        return grade, "Basic", "Public", "None", "None"
    elif 19 <= age <= 23:
        major = np.random.choice(MAJORS, p=MAJOR_PROBS)
        return "University Student", "Higher Ed", "None", major, "Public"
    else:
        return "Graduate/Post-Graduate", "Higher Ed", "None", "STAY_SAME", "None"

def extract_and_save_relational_data(df, is_new_batch, year):
    """Generates all 6 relational files simultaneously."""
    if df.empty: return

    # Dimensions (Only updated for new arrivals to save memory)
    if is_new_batch:
        dim_resident = df[['National_ID', 'Gender', 'Year_of_Birth', 'Year_of_Death']].drop_duplicates()
        append_to_csv(dim_resident, 'Dim_Resident.csv')
        dim_re = df[['Real_Estate_ID', 'Residence_Site', 'Residence_Type', 'Household_Green_Space_sqm']].drop_duplicates()
        append_to_csv(dim_re, 'Dim_Real_Estate.csv')

    # Fact Tables
    df_facts = df.copy()
    df_facts['Report_Year'] = year
    df_facts['Record_ID'] = [uuid.uuid4() for _ in range(len(df_facts))]

    # 1. Fact_SocioEconomic
    append_to_csv(df_facts[['Record_ID', 'Report_Year', 'National_ID', 'Real_Estate_ID', 'Current_Age', 
                            'Marital_Status', 'Career', 'Income_per_Year_EGP', 'Primary_Transport_Type']], 
                  'Fact_SocioEconomic.csv')
    
    # 2. Fact_Education
    append_to_csv(df_facts[['Record_ID', 'Report_Year', 'National_ID', 'Edu_Grade', 'Edu_Type', 
                            'School_Type', 'University_Major', 'University_Type']], 
                  'Fact_Education.csv')
    
    # 3. Fact_Health_Medical
    append_to_csv(df_facts[['Record_ID', 'Report_Year', 'National_ID', 'Health_Percent', 'Chronic_Disease', 
                            'Hospital_Service_Needed', 'Requires_Hospital_Bed', 'BMI']], 
                  'Fact_Health_Medical.csv')
    
    # 4. Fact_Household_Environment
    # Note: Aggregate by Real_Estate_ID to get household-level data
    env_cols = ['Real_Estate_ID', 'Report_Year', 'Has_Solar_System', 'Has_AC', 'Uses_Greywater_Recycling', 
                'Avg_Monthly_Electricity_KW', 'Carbon_Footprint_Index']
    fact_env = df_facts[env_cols].drop_duplicates(subset=['Real_Estate_ID'])
    append_to_csv(fact_env, 'Fact_Household_Environment.csv')

def run_simulation():
    print("🔥 Starting Script v2.2: Full Data Regeneration...")
    living_population_df = pd.DataFrame()
    
    # Clean previous run
    if os.path.exists(OUTPUT_DIR):
        for f in os.listdir(OUTPUT_DIR): os.remove(os.path.join(OUTPUT_DIR, f))

    for year, target_pop in POPULATION_TARGETS.items():
        print(f"> Processing {year}...")
        
        if not living_population_df.empty:
            # 1. Ageing & Education Logic Fix
            living_population_df['Current_Age'] += 5
            for idx, row in living_population_df.iterrows():
                grade, etype, stype, major, utype = assign_education_and_major(row['Current_Age'])
                living_population_df.at[idx, 'Edu_Grade'] = grade
                living_population_df.at[idx, 'Edu_Type'] = etype
                # RELATIONAL SYNC: Only overwrite major if they just started University
                if major != "STAY_SAME" and major != "None":
                    living_population_df.at[idx, 'University_Major'] = major

            # 2. Career Progression (Major -> Career)
            mask_workers = (living_population_df['Current_Age'] >= 22) & (living_population_df['Career'] == 'Unemployed/Student')
            if mask_workers.any():
                # Careers now inherit from the Major column in Education
                living_population_df.loc[mask_workers, 'Career'] = living_population_df.loc[mask_workers, 'University_Major']
                living_population_df.loc[mask_workers, 'Income_per_Year_EGP'] = [
                    calculate_skewed_income(c) for c in living_population_df.loc[mask_workers, 'Career']
                ]

            # 3. Mortality
            living_population_df = living_population_df[living_population_df['Year_of_Death'] > year]
            extract_and_save_relational_data(living_population_df, is_new_batch=False, year=year)

        # 4. Handle Growth
        current_pop = len(living_population_df)
        shortfall = target_pop - current_pop
        if shortfall > 0:
            new_arrivals = generate_population_batch(year, shortfall) # Assume helper is present
            extract_and_save_relational_data(new_arrivals, is_new_batch=True, year=year)
            living_population_df = pd.concat([living_population_df, new_arrivals], ignore_index=True)

    print("✅ All 6 files generated successfully in /nac_relational_dataset")

if __name__ == "__main__":
    run_simulation()

Starting Fully Patched & Collision-Free NAC Simulation (2025-2050)...

--- Processing Year 2025 ---
Generating 350,000 new residents to reach 350,000 target...
Exporting Dimensions and Fact snapshots for new arrivals...

--- Processing Year 2030 ---
Recorded 127 deaths since last period.
Exporting Fact snapshots for surviving population...
Generating 743,710 new residents to reach 1,050,000 target...
Exporting Dimensions and Fact snapshots for new arrivals...

--- Processing Year 2035 ---
Recorded 584 deaths since last period.
Exporting Fact snapshots for surviving population...
Generating 994,517 new residents to reach 1,950,000 target...
Exporting Dimensions and Fact snapshots for new arrivals...

--- Processing Year 2040 ---
Recorded 2,032 deaths since last period.
Exporting Fact snapshots for surviving population...
Generating 1,277,142 new residents to reach 3,100,000 target...
Exporting Dimensions and Fact snapshots for new arrivals...

--- Processing Year 2045 ---
Recorded 5,732